In [32]:

%pip install pandas numpy tabulate --quiet

print("✅ Dependencies installed.")


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ Dependencies installed.


In [33]:

import os
import json
import math
import warnings
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')

import numpy  as np
import pandas as pd
from tabulate import tabulate   # Clean table rendering for scorecard display

print("✅ All imports successful.")

✅ All imports successful.


In [34]:
# ── Entity & Paths ────────────────────────────────────────────
entity_name      = "Tesla"   # <-- must match Phase 1 & 2
PHASE2_CSV_PATH  = f"adverse_detection_{entity_name.replace(' ', '_')}_phase2.csv"
OUTPUT_CSV_PATH  = f"risk_scored_{entity_name.replace(' ', '_')}_phase3.csv"
OUTPUT_JSON_PATH = f"risk_scorecard_{entity_name.replace(' ', '_')}_phase3.json"

# ── Dimension Weights (must sum to 100) ───────────────────────
# These control how much each signal contributes to the
# per-article risk score. Adjust based on your risk framework.
WEIGHTS = {
    "category_severity" : 40,   # Risk category danger level
    "llm_confidence"    : 25,   # Model certainty in the adverse flag
    "source_credibility": 20,   # Publisher trustworthiness
    "recency"           : 15,   # How recent the article is
}
assert sum(WEIGHTS.values()) == 100, "Weights must sum to 100"

# ── Category Severity Scores (0.0 – 1.0) ─────────────────────
# Higher = more severe. These reflect standard compliance risk
# frameworks (FATF, Basel AML, FCA guidance).
# "Not Adverse" = 0.0 so non-adverse articles contribute nothing.
CATEGORY_SEVERITY = {
    "Sanctions & Watchlist"  : 1.00,
    "Fraud & Financial Crime": 0.95,
    "Corruption & Bribery"   : 0.90,
    "Legal & Regulatory"     : 0.80,
    "Cybersecurity & Data"   : 0.75,
    "Labor & Human Rights"   : 0.70,
    "ESG & Environmental"    : 0.65,
    "Operational Risk"       : 0.60,
    "Reputational"           : 0.40,
    "Not Adverse"            : 0.00,
}

# ── Source Credibility Scores (0.0 – 1.0) ────────────────────
# Tier 1: Major newswires and financial press (highest weight)
# Tier 2: Established regional/specialist outlets
# Tier 3: Blogs, aggregators, unknown sources (lowest weight)
#
# Add sources here as your pipeline encounters new ones.
# Sources not listed default to UNKNOWN_SOURCE_SCORE.
SOURCE_CREDIBILITY = {
    # ── Tier 1: Premium financial/news wire ──
    "bloomberg"         : 1.00,
    "bloomberg.com"     : 1.00,
    "reuters"           : 1.00,
    "the wall street journal": 0.98,
    "wsj"               : 0.98,
    "financial times"   : 0.98,
    "ft"                : 0.98,
    "the new york times": 0.95,
    "nyt"               : 0.95,
    "associated press"  : 0.95,
    "ap"                : 0.95,
    "bbc"               : 0.93,
    "bbc news"          : 0.93,

    # ── Tier 2: Established financial/tech press ──
    "the guardian"      : 0.88,
    "guardian"          : 0.88,
    "washington post"   : 0.88,
    "los angeles times" : 0.85,
    "fortune"           : 0.85,
    "forbes"            : 0.82,
    "cnbc"              : 0.82,
    "yahoo finance"     : 0.75,
    "barron's"          : 0.80,
    "marketwatch"       : 0.78,
    "techcrunch"        : 0.75,
    "the verge"         : 0.73,
    "wired"             : 0.73,
    "ars technica"      : 0.73,
    "king5.com"         : 0.65,
    "ktla"              : 0.65,
    "edmunds"           : 0.68,
    "motor1.com"        : 0.60,
    "autoblog"          : 0.60,
    "electrek"          : 0.60,
    "the drive"         : 0.58,
    "sherwood news"     : 0.55,
    "cleantechnica"     : 0.55,

    # ── Tier 3: Aggregators / unknown blogs ──
    "yahoo autos"                  : 0.50,
    "yahoo finance singapore"      : 0.50,
    "newsable_asianetnews"         : 0.35,
    "economictimes_indiatimes"     : 0.45,
    "sedaily"                      : 0.35,
    "fivepaisa"                    : 0.30,
    "newspub_live"                 : 0.25,
    "itbrief_co_nz"                : 0.30,
    "fool"                         : 0.50,   # Motley Fool
    "google news"                  : 0.40,   # Aggregator
    "bing news"                    : 0.40,
}
UNKNOWN_SOURCE_SCORE = 0.30   # Default for any source not listed above

# ── Recency Decay Config ──────────────────────────────────────
# Articles decay in scoring weight as they age.
# RECENCY_HALF_LIFE_DAYS: number of days after which an article's
# recency score is halved. Default: 30 days.
#
# Example with half_life=30:
#   0 days old  → recency score = 1.00
#   30 days old → recency score = 0.50
#   60 days old → recency score = 0.25
#   90 days old → recency score = 0.125
RECENCY_HALF_LIFE_DAYS = 30
RECENCY_FLOOR          = 0.05   # Minimum recency score (very old articles still count a little)

# ── Entity Score Aggregation ──────────────────────────────────
# Controls how per-article scores roll up to one entity score.
#
# AGGREGATION_METHOD options:
#   "diminishing"  → each additional article adds less (recommended)
#                    prevents score inflation from duplicate coverage
#   "weighted_avg" → simple weighted average of per-article scores
#   "max"          → entity score = score of the single worst article
AGGREGATION_METHOD = "diminishing"

# Diminishing returns base: score_n = score_n * (DIMINISHING_BASE ^ (n-1))
# 0.75 means each additional article contributes 75% of the previous one.
DIMINISHING_BASE = 0.75

# ── Risk Label Thresholds ─────────────────────────────────────
# Entity score → human-readable risk label
RISK_THRESHOLDS = {
    "Critical": 85,
    "High": 65,
    "Medium": 35,
    "Low": 0
}
RISK_LABEL_EMOJI = {
    "Critical": "🔴",
    "High"    : "🟠",
    "Medium"  : "🟡",
    "Low"     : "🟢",
}

print("✅ Configuration loaded.")
print(f"   Entity             : {entity_name}")
print(f"   Aggregation method : {AGGREGATION_METHOD}")
print(f"   Dimension weights  : {WEIGHTS}")
print(f"   Risk thresholds    : Critical≥{RISK_THRESHOLDS['Critical']} | "
      f"High≥{RISK_THRESHOLDS['High']} | "
      f"Medium≥{RISK_THRESHOLDS['Medium']}")

✅ Configuration loaded.
   Entity             : Tesla
   Aggregation method : diminishing
   Dimension weights  : {'category_severity': 40, 'llm_confidence': 25, 'source_credibility': 20, 'recency': 15}
   Risk thresholds    : Critical≥85 | High≥65 | Medium≥35


In [35]:

def load_phase2_data(csv_path: str) -> pd.DataFrame:
    """
    Loads Phase 2 adverse detection output.

    Priority:
      1. In-memory `df_analyzed` from the same Jupyter session
      2. CSV saved by Phase 2 at `csv_path`

    Validates that all required columns are present before returning.
    """
    required_cols = [
        "title", "source", "publication_date", "article_url",
        "is_adverse", "adverse_category", "confidence_score", "reason"
    ]

    # --- Try in-memory DataFrame first ---
    try:
        ipython = get_ipython()
        if 'df_analyzed' in ipython.user_ns:
            df = ipython.user_ns['df_analyzed']
            if isinstance(df, pd.DataFrame) and not df.empty:
                missing = [c for c in required_cols if c not in df.columns]
                if not missing:
                    print(f"✅ Loaded df_analyzed from session memory ({len(df)} articles).")
                    return df.copy()
    except Exception:
        pass

    # --- Fall back to CSV ---
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise ValueError(f"Phase 2 CSV missing columns: {missing}")
        print(f"✅ Loaded Phase 2 data from CSV: '{csv_path}' ({len(df)} articles).")
        return df

    raise FileNotFoundError(
        f"Phase 2 data not found in memory or at '{csv_path}'.\n"
        f"Please run Phase 2 first."
    )




def score_category_severity(adverse_category: str) -> float:
    """
    Returns a 0.0–1.0 severity score for the risk category.

    Looks up the category in CATEGORY_SEVERITY.
    Unknown categories default to the 'Reputational' score (0.40)
    as a conservative fallback.

    Called with: the `adverse_category` column from Phase 2.
    """
    return CATEGORY_SEVERITY.get(adverse_category, 0.40)


def score_llm_confidence(confidence_score: float) -> float:
    """
    Returns the LLM confidence score as-is (already 0.0–1.0).

    Clamps to [0, 1] to handle any edge cases from Phase 2.
    A confidence of 0.90 contributes 0.90 * WEIGHTS['llm_confidence']
    to the final per-article score.
    """
    return float(max(0.0, min(1.0, confidence_score)))


def score_source_credibility(source: str) -> float:
    """
    Returns a 0.0–1.0 credibility score for the news source.

    Matching strategy:
      1. Exact match (case-insensitive) against SOURCE_CREDIBILITY
      2. Substring match — e.g. 'bloomberg.com' matches 'bloomberg'
      3. Fallback to UNKNOWN_SOURCE_SCORE

    This handles variations in how sources are named across
    Google News RSS vs NewsData API vs Bing RSS.
    """
    if not source or pd.isna(source):
        return UNKNOWN_SOURCE_SCORE

    source_lower = str(source).lower().strip()

    # Exact match
    if source_lower in SOURCE_CREDIBILITY:
        return SOURCE_CREDIBILITY[source_lower]

    # Substring match — check if any key appears in the source string
    for key, score in SOURCE_CREDIBILITY.items():
        if key in source_lower or source_lower in key:
            return score

    return UNKNOWN_SOURCE_SCORE


def score_recency(publication_date: str) -> float:
    """
    Returns a 0.0–1.0 recency score using exponential decay.

    Formula: score = max(FLOOR, 2 ^ (-age_days / HALF_LIFE))

    This is the same decay model used in nuclear physics and
    information retrieval — intuitively, an article's relevance
    halves every RECENCY_HALF_LIFE_DAYS days.

    Articles with unparseable dates get a mid-range score of 0.50
    (we don't want to penalize them for a data quality issue).
    """
    if not publication_date or publication_date == 'Unknown' or pd.isna(publication_date):
        return 0.50   # Unknown date → neutral recency

    try:
        pub_dt   = pd.to_datetime(publication_date)
        today    = pd.Timestamp.now().normalize()
        age_days = max(0, (today - pub_dt).days)

        # Exponential decay: halves every RECENCY_HALF_LIFE_DAYS days
        decay_score = math.pow(2, -age_days / RECENCY_HALF_LIFE_DAYS)

        return max(RECENCY_FLOOR, decay_score)
    except Exception:
        return 0.50




def compute_article_risk_score(row: pd.Series) -> dict:
    """
    Computes a per-article risk score from all four signal dimensions.

    Only adverse articles receive a non-zero score.
    Non-adverse articles return score=0.0 with all sub-scores=0.0.

    Returns a dict with:
      article_risk_score  : float, 0–100, the final weighted score
      score_category      : float, 0–100, category severity contribution
      score_confidence    : float, 0–100, LLM confidence contribution
      score_source        : float, 0–100, source credibility contribution
      score_recency       : float, 0–100, recency contribution
      raw_*               : float, 0–1, the raw signal before weighting
    """
    # Non-adverse articles contribute nothing to entity risk
    if not row.get('is_adverse', False):
        return {
            "article_risk_score": 0.0,
            "score_category"    : 0.0,
            "score_confidence"  : 0.0,
            "score_source"      : 0.0,
            "recency_points"     : 0.0,
            "raw_category"      : 0.0,
            "raw_confidence"    : 0.0,
            "raw_source"        : 0.0,
            "raw_recency"       : 0.0,
        }

    # ── Compute raw signal scores (0.0 – 1.0) ────────────────
    raw_category   = score_category_severity(row.get('adverse_category', 'Not Adverse'))
    raw_confidence = score_llm_confidence(row.get('confidence_score', 0.0))
    raw_source     = score_source_credibility(row.get('source', ''))
    raw_recency    = score_recency(row.get('publication_date', ''))

    # ── Weight each signal by its configured weight ───────────
    # Each dimension contributes: raw_score * weight (in points out of 100)
    score_category   = raw_category   * WEIGHTS["category_severity"]
    score_confidence = raw_confidence * WEIGHTS["llm_confidence"]
    score_source     = raw_source     * WEIGHTS["source_credibility"]
    recency_points   = raw_recency    * WEIGHTS["recency"]

    # ── Total per-article score (0–100) ───────────────────────
    article_risk_score = score_category + score_confidence + score_source + recency_points
    article_risk_score = round(min(100.0, max(0.0, article_risk_score)), 2)

    return {
        "article_risk_score": article_risk_score,
        "score_category"    : round(score_category, 2),
        "score_confidence"  : round(score_confidence, 2),
        "score_source"      : round(score_source, 2),
        "recency_points"     : round(recency_points, 2),
        "raw_category"      : round(raw_category, 3),
        "raw_confidence"    : round(raw_confidence, 3),
        "raw_source"        : round(raw_source, 3),
        "raw_recency"       : round(raw_recency, 3),
    }


def score_all_articles(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies compute_article_risk_score to every row in df.

    Returns df with 9 new columns appended:
      article_risk_score, score_category, score_confidence,
      score_source, score_recency,
      raw_category, raw_confidence, raw_source, raw_recency
    """
    print(f"  Scoring {len(df)} articles across 4 signal dimensions...")

    score_records = [compute_article_risk_score(row) for _, row in df.iterrows()]
    scores_df     = pd.DataFrame(score_records)

    df_out = df.copy().reset_index(drop=True)
    for col in scores_df.columns:
        df_out[col] = scores_df[col].values

    return df_out



def aggregate_entity_score(df_scored: pd.DataFrame) -> float:
    """
    Rolls up all per-article scores into a single entity-level score.

    Three methods are available (controlled by AGGREGATION_METHOD):

    1. 'diminishing' (recommended):
       Articles are sorted by score descending.
       Each article contributes: score_i * (DIMINISHING_BASE ^ i)
       where i is the article's rank (0-indexed).

       This models the real-world compliance insight that the 2nd
       article about the same event adds marginal new risk —
       both Bloomberg and Reuters covering the same crash doesn't
       mean the entity is twice as risky.

       The result is then normalized to 0–100.

    2. 'weighted_avg':
       Simple weighted average of non-zero article scores.
       Better when articles cover truly different events.

    3. 'max':
       Entity score = the worst single article score.
       Conservative — useful for zero-tolerance screening.

    Returns:
        float: Entity risk score, 0.0–100.0
    """
    # Only consider adverse articles
    adverse_scores = df_scored[
        df_scored['is_adverse'] == True
    ]['article_risk_score'].sort_values(ascending=False).tolist()

    if not adverse_scores:
        return 0.0

    if AGGREGATION_METHOD == "max":
        return round(adverse_scores[0], 2)

    if AGGREGATION_METHOD == "weighted_avg":
        return round(float(np.mean(adverse_scores)), 2)

    if AGGREGATION_METHOD == "diminishing":
        # Diminishing returns sum
        diminished = sum(
            score * (DIMINISHING_BASE ** i)
            for i, score in enumerate(adverse_scores)
        )
        # Normalize: theoretical max is score_max * (1 / (1 - BASE))
        # For BASE=0.75, max multiplier = 4.0
        theoretical_max = 100.0 * (1.0 / (1.0 - DIMINISHING_BASE))
        normalized      = (diminished / theoretical_max) * 100.0
        return round(min(100.0, normalized), 2)

    raise ValueError(f"Unknown AGGREGATION_METHOD: {AGGREGATION_METHOD}")


def get_risk_label(score: float) -> str:
    """
    Maps a numeric entity score to a risk label string.
    Uses RISK_THRESHOLDS from config.
    """
    for label, threshold in sorted(RISK_THRESHOLDS.items(), key=lambda x: -x[1]):
        if score >= threshold:
            return label
    return "Low"



def build_risk_scorecard(entity: str, df_scored: pd.DataFrame) -> dict:
    """
    Builds the full structured risk scorecard for the entity.

    The scorecard is the primary output of Phase 3 and the
    primary input to Phase 4 (RAG Copilot) and Phase 5 (Report).

    Structure:
    {
      entity_name          : str
      screening_date       : str (ISO 8601)
      entity_risk_score    : float (0–100)
      risk_label           : str (Low/Medium/High/Critical)
      risk_emoji           : str
      aggregation_method   : str
      total_articles       : int
      adverse_articles     : int
      non_adverse_articles : int
      adverse_rate_pct     : float
      category_breakdown   : dict  { category: { count, avg_score, max_score } }
      source_breakdown     : dict  { source: { count, avg_credibility } }
      dimension_scores     : dict  { dimension: weighted_avg_contribution }
      contributing_articles: list  [ { title, source, date, category,
                                       confidence, article_risk_score, reason } ]
      weights_used         : dict
      thresholds_used      : dict
    }
    """
    adverse_df     = df_scored[df_scored['is_adverse'] == True].copy()
    entity_score   = aggregate_entity_score(df_scored)
    risk_label     = get_risk_label(entity_score)

    # ── Category Breakdown ────────────────────────────────────
    category_breakdown = {}
    if not adverse_df.empty:
        for cat, grp in adverse_df.groupby('adverse_category'):
            category_breakdown[cat] = {
                "count"    : int(len(grp)),
                "avg_score": round(grp['article_risk_score'].mean(), 2),
                "max_score": round(grp['article_risk_score'].max(), 2),
                "severity" : CATEGORY_SEVERITY.get(cat, 0.40),
            }

    # ── Source Breakdown (adverse articles only) ──────────────
    source_breakdown = {}
    if not adverse_df.empty:
        for src, grp in adverse_df.groupby('source'):
            source_breakdown[str(src)] = {
                "count"           : int(len(grp)),
                "avg_credibility" : round(grp['raw_source'].mean(), 3),
                "avg_article_score": round(grp['article_risk_score'].mean(), 2),
            }

    # ── Dimension Score Contributions (avg across adverse articles) ──
    dimension_scores = {}
    if not adverse_df.empty:
        dimension_scores = {
            "category_severity" : round(adverse_df['score_category'].mean(), 2),
            "llm_confidence"    : round(adverse_df['score_confidence'].mean(), 2),
            "source_credibility": round(adverse_df['score_source'].mean(), 2),
            "recency"           : round(adverse_df['recency_points'].mean(), 2),
        }

    # ── Contributing Articles (sorted by article_risk_score desc) ──
    contributing_articles = []
    if not adverse_df.empty:
        for _, row in adverse_df.sort_values('article_risk_score', ascending=False).iterrows():
            contributing_articles.append({
                "title"             : str(row.get('title', '')),
                "source"            : str(row.get('source', '')),
                "publication_date"  : str(row.get('publication_date', '')),
                "article_url"       : str(row.get('article_url', '')),
                "adverse_category"  : str(row.get('adverse_category', '')),
                "confidence_score"  : float(row.get('confidence_score', 0.0)),
                "article_risk_score": float(row.get('article_risk_score', 0.0)),
                "score_breakdown"   : {
                    "category_severity" : float(row.get('score_category', 0.0)),
                    "llm_confidence"    : float(row.get('score_confidence', 0.0)),
                    "source_credibility": float(row.get('score_source', 0.0)),
                    "recency"           : float(row.get('recency_points', 0.0)),
                },
                "reason"            : str(row.get('reason', '')),
                "detection_layer"   : str(row.get('detection_layer', '')),
            })

    # ── Assemble Full Scorecard ───────────────────────────────
    scorecard = {
        "entity_name"         : entity,
        "screening_date"      : datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
        "entity_risk_score"   : entity_score,
        "risk_label"          : risk_label,
        "risk_emoji"          : RISK_LABEL_EMOJI.get(risk_label, "❓"),
        "aggregation_method"  : AGGREGATION_METHOD,
        "total_articles"      : int(len(df_scored)),
        "adverse_articles"    : int(len(adverse_df)),
        "non_adverse_articles": int(len(df_scored) - len(adverse_df)),
        "adverse_rate_pct"    : round(100 * len(adverse_df) / max(len(df_scored), 1), 1),
        "category_breakdown"  : category_breakdown,
        "source_breakdown"    : source_breakdown,
        "dimension_scores"    : dimension_scores,
        "contributing_articles": contributing_articles,
        "weights_used"        : WEIGHTS,
        "thresholds_used"     : RISK_THRESHOLDS,
    }

    return scorecard



def run_risk_scoring(entity: str) -> tuple:
    """
    Master orchestrator for Phase 3.

    Steps:
      1. Load Phase 2 output (df_analyzed)
      2. Score every article across 4 dimensions
      3. Aggregate to entity-level score
      4. Build full risk scorecard dict
      5. Save df_scored to CSV and scorecard to JSON

    Returns:
        tuple: (df_scored: pd.DataFrame, risk_scorecard: dict)
    """
    print(f"\n📊 Starting Phase 3 Risk Scoring for entity: '{entity}'")
    print("-" * 60)

    # Step 1: Load
    df = load_phase2_data(PHASE2_CSV_PATH)

    # Step 2: Per-article scoring
    df_scored = score_all_articles(df)

    # Step 3 & 4: Aggregate + build scorecard
    print(f"  Aggregating entity score (method: {AGGREGATION_METHOD})...")
    risk_scorecard = build_risk_scorecard(entity, df_scored)

    # Step 5: Save outputs
    df_scored.to_csv(OUTPUT_CSV_PATH, index=False)
    with open(OUTPUT_JSON_PATH, 'w', encoding='utf-8') as f:
        json.dump(risk_scorecard, f, indent=2, ensure_ascii=False)

    print("-" * 60)
    print(f"✅ Scoring complete.")
    print(f"💾 CSV  saved → '{OUTPUT_CSV_PATH}'")
    print(f"💾 JSON saved → '{OUTPUT_JSON_PATH}'")

    return df_scored, risk_scorecard


print("✅ All scoring functions defined.")

✅ All scoring functions defined.


In [36]:

df_scored, risk_scorecard = run_risk_scoring(entity_name)


📊 Starting Phase 3 Risk Scoring for entity: 'Tesla'
------------------------------------------------------------
✅ Loaded Phase 2 data from CSV: 'adverse_detection_Tesla_phase2.csv' (30 articles).
  Scoring 30 articles across 4 signal dimensions...
  Aggregating entity score (method: diminishing)...
------------------------------------------------------------
✅ Scoring complete.
💾 CSV  saved → 'risk_scored_Tesla_phase3.csv'
💾 JSON saved → 'risk_scorecard_Tesla_phase3.json'


In [37]:


sc = risk_scorecard   # alias for readability
emoji = sc['risk_emoji']
label = sc['risk_label']
score = sc['entity_risk_score']

# ── Section 1: Entity Verdict ─────────────────────────────────
print("=" * 70)
print(f"  PHASE 3 RISK SCORECARD — {sc['entity_name'].upper()}")
print("=" * 70)
print(f"  Screening Date       : {sc['screening_date']}")
print(f"  Total Articles       : {sc['total_articles']}")
print(f"  Adverse Articles     : {sc['adverse_articles']} ({sc['adverse_rate_pct']}%)")
print(f"  Aggregation Method   : {sc['aggregation_method']}")
print()
print(f"  ┌─────────────────────────────────────────┐")
print(f"  │  Entity Risk Score  :  {score:<6.1f} / 100       │")
print(f"  │  Risk Label         :  {emoji} {label:<20}  │")
print(f"  └─────────────────────────────────────────┘")

# Visual score bar
filled = int(score / 5)   # 20 chars = 100 pts
bar    = "█" * filled + "░" * (20 - filled)
print(f"\n  Score Bar : [{bar}] {score:.1f}/100")

# ── Section 2: Dimension Score Contributions ──────────────────
print("\n" + "-" * 70)
print("  📐 SCORING DIMENSION CONTRIBUTIONS (avg across adverse articles)")
print("-" * 70)
if sc['dimension_scores']:
    dim_rows = []
    for dim, contrib in sc['dimension_scores'].items():
        weight     = WEIGHTS.get(dim, 0)
        max_pts    = weight
        pct_of_max = (contrib / max_pts * 100) if max_pts > 0 else 0
        dim_bar    = "█" * int(pct_of_max / 5) + "░" * (20 - int(pct_of_max / 5))
        dim_rows.append([
            dim.replace('_', ' ').title(),
            f"{weight} pts",
            f"{contrib:.2f} pts",
            f"{pct_of_max:.1f}%",
            dim_bar
        ])
    print(tabulate(
        dim_rows,
        headers=["Dimension", "Max Weight", "Avg Contribution", "% Utilized", "Bar"],
        tablefmt="simple"
    ))
else:
    print("  No adverse articles — all dimension scores are 0.")

# ── Section 3: Category Breakdown ─────────────────────────────
print("\n" + "-" * 70)
print("  📌 RISK CATEGORY BREAKDOWN")
print("-" * 70)
if sc['category_breakdown']:
    cat_rows = []
    for cat, stats in sorted(
        sc['category_breakdown'].items(),
        key=lambda x: -x[1]['max_score']
    ):
        cat_rows.append([
            cat,
            stats['count'],
            f"{stats['severity']:.2f}",
            f"{stats['avg_score']:.1f}",
            f"{stats['max_score']:.1f}",
        ])
    print(tabulate(
        cat_rows,
        headers=["Category", "Articles", "Severity", "Avg Score", "Max Score"],
        tablefmt="simple"
    ))
else:
    print("  No adverse categories detected.")

# ── Section 4: Source Breakdown ───────────────────────────────
print("\n" + "-" * 70)
print("  📰 SOURCE BREAKDOWN (adverse articles only)")
print("-" * 70)
if sc['source_breakdown']:
    src_rows = []
    for src, stats in sorted(
        sc['source_breakdown'].items(),
        key=lambda x: -x[1]['avg_article_score']
    ):
        src_rows.append([
            src,
            stats['count'],
            f"{stats['avg_credibility']:.2f}",
            f"{stats['avg_article_score']:.1f}",
        ])
    print(tabulate(
        src_rows,
        headers=["Source", "Articles", "Credibility", "Avg Risk Score"],
        tablefmt="simple"
    ))

# ── Section 5: Per-Article Risk Scores ───────────────────────
print("\n" + "-" * 70)
print("  📋 PER-ARTICLE RISK SCORES (adverse articles, ranked)")
print("-" * 70)

adverse_scored = df_scored[
    df_scored['is_adverse'] == True
].sort_values('article_risk_score', ascending=False).reset_index(drop=True)

if not adverse_scored.empty:
    art_rows = []
    for i, row in adverse_scored.iterrows():
        title_short = str(row['title'])[:55] + '...' if len(str(row['title'])) > 55 else str(row['title'])
        art_rows.append([
            i + 1,
            title_short,
            row['source'],
            row['publication_date'],
            row['adverse_category'],
            f"{row['article_risk_score']:.1f}",
            f"{row['score_category']:.1f}+{row['score_confidence']:.1f}+{row['score_source']:.1f}+{row['recency_points']:.1f}",
        ])
    print(tabulate(
        art_rows,
        headers=["#", "Title", "Source", "Date", "Category", "Score", "Cat+Conf+Src+Rec"],
        tablefmt="simple"
    ))

    # Detailed breakdown for top article
    print("\n  🔎 Top Article — Full Score Breakdown:")
    print("  " + "-" * 50)
    top = adverse_scored.iloc[0]
    print(f"  Title      : {top['title']}")
    print(f"  Source     : {top['source']} (credibility: {top['raw_source']:.2f})")
    print(f"  Date       : {top['publication_date']} (recency raw: {top['raw_recency']:.3f})")
    print(f"  Category   : {top['adverse_category']} (severity: {top['raw_category']:.2f})")
    print(f"  Confidence : {top['raw_confidence']:.2f}")
    print(f"  ── Score breakdown (pts) ──")
    print(f"    Category Severity  : {top['raw_category']:.2f} × {WEIGHTS['category_severity']} = {top['score_category']:.2f}")
    print(f"    LLM Confidence     : {top['raw_confidence']:.2f} × {WEIGHTS['llm_confidence']}  = {top['score_confidence']:.2f}")
    print(f"    Source Credibility : {top['raw_source']:.2f} × {WEIGHTS['source_credibility']} = {top['score_source']:.2f}")
    print(f"    Recency            : {top['raw_recency']:.3f} × {WEIGHTS['recency']}  = {top['recency_points']:.2f}")
    print(f"    ─────────────────────────────────────")
    print(f"    Article Risk Score : {top['article_risk_score']:.2f} / 100")
    print(f"  Reason     : {top['reason']}")
else:
    print("  No adverse articles to display.")

# ── Final Summary ─────────────────────────────────────────────
print("\n" + "=" * 70)
print(f"  ✅ Phase 3 complete.")
print(f"     Entity Risk Score : {score:.1f} / 100")
print(f"     Risk Label        : {emoji} {label}")
print(f"     CSV  → '{OUTPUT_CSV_PATH}'")
print(f"     JSON → '{OUTPUT_JSON_PATH}'")
print(f"     → Phase 4 (RAG Copilot) loads both files.")
print("=" * 70)

  PHASE 3 RISK SCORECARD — TESLA
  Screening Date       : 2026-06-14T15:44:16Z
  Total Articles       : 30
  Adverse Articles     : 4 (13.3%)
  Aggregation Method   : diminishing

  ┌─────────────────────────────────────────┐
  │  Entity Risk Score  :  52.2   / 100       │
  │  Risk Label         :  🟡 Medium                │
  └─────────────────────────────────────────┘

  Score Bar : [██████████░░░░░░░░░░] 52.2/100

----------------------------------------------------------------------
  📐 SCORING DIMENSION CONTRIBUTIONS (avg across adverse articles)
----------------------------------------------------------------------
Dimension           Max Weight    Avg Contribution    % Utilized    Bar
------------------  ------------  ------------------  ------------  --------------------
Category Severity   40 pts        24.00 pts           60.0%         ████████████░░░░░░░░
Llm Confidence      25 pts        21.88 pts           87.5%         █████████████████░░░
Source Credibility  20 pts      